In [29]:
import numpy as np
import pandas as pd

# Set seed for reproducibility
np.random.seed(101)
n_seconds = 1000

# 1. Generate realistic timestamps (1 measurement per second)
timestamps = pd.date_range(start='2026-11-09 10:00:00', periods=n_seconds, freq='s')

# 2. Simulate base temperature curve using NumPy (gradual heat up + random sensor noise)
time_index = np.arange(n_seconds)
base_temp = 40 + 25 * np.sin(time_index / 150) + np.random.normal(0, 1.5, n_seconds)

# 3. Simulate voltage (Volts) and current (Amperes)
voltage = np.random.normal(1.20, 0.03, n_seconds)
current = 15 + 5 * np.sin(time_index / 150) + np.random.normal(0, 0.4, n_seconds)

# Inject a real hardware anomaly (Thermal & Voltage Spike between 450s and 480s)
base_temp[450:480] += 22.0
voltage[450:480] -= 0.18

# Combine into a Pandas DataFrame
df_sensor = pd.DataFrame({
    'timestamp': timestamps,
    'second': time_index,
    'sensor_temp_C': base_temp,
    'core_voltage_V': voltage,
    'core_current_A': current
})

df_sensor.head()


,timestamp,second,sensor_temp_C,core_voltage_V,core_current_A
0,2026-11-09 10:00:00,0,44.060275,1.219549,14.393320
1,2026-11-09 10:00:01,1,41.108864,1.197757,15.500292
2,2026-11-09 10:00:02,2,41.695278,1.188463,15.430607
3,2026-11-09 10:00:03,3,41.255705,1.213128,15.073413
4,2026-11-09 10:00:04,4,41.643265,1.184728,15.049772


In [30]:
# --- DATA MANIPULATION PIPELINE ---

# 1. Feature Engineering: Calculate Power Consumption in Watts (P = V * I)
df_sensor['power_watt'] = df_sensor['core_voltage_V'] * df_sensor['core_current_A']

# 2. Time-Series Smoothing: 10-second rolling average to remove sensor jitter
df_sensor['smoothed_temp_C'] = df_sensor['sensor_temp_C'].rolling(window=10).mean()

# 3. Conditional Filtering: Isolate critical overheating events (> 75°C)
overheating_events = df_sensor[df_sensor['sensor_temp_C'] > 75.0]

print(f"Total Critical Overheating Seconds Detected: {len(overheating_events)} seconds")
print(f"Peak Power Consumption Reached: {df_sensor['power_watt'].max():.2f} Watts")

# Preview the manipulated dataset with new columns
df_sensor.iloc[448:455]


Total Critical Overheating Seconds Detected: 0 seconds
Peak Power Consumption Reached: 26.26 Watts


,timestamp,second,sensor_temp_C,core_voltage_V,core_current_A,power_watt,smoothed_temp_C
448,2026-11-09 10:07:28,448,43.536287,1.186305,16.367970,19.417402,44.706444
449,2026-11-09 10:07:29,449,44.180413,1.178209,15.656503,18.446629,44.480001
450,2026-11-09 10:07:30,450,63.998939,1.065211,15.647501,16.667895,46.198070
451,2026-11-09 10:07:31,451,63.497897,1.042317,15.464115,16.118505,48.201110
452,2026-11-09 10:07:32,452,64.647677,1.002394,16.235823,16.274697,50.011817
453,2026-11-09 10:07:33,453,65.916336,1.032681,15.891537,16.410884,52.273311
454,2026-11-09 10:07:34,454,68.796331,0.993126,15.594812,15.487611,54.895102


In [31]:
import plotly.graph_objects as go

fig = go.Figure()

# Ham sensör verisi
fig.add_trace(go.Scatter(
    x=df_sensor['timestamp'],
    y=df_sensor['sensor_temp_C'],
    mode='lines',
    name='Raw Sensor Temp (°C)',
    line=dict(color='rgba(150, 150, 150, 0.5)', width=1.5)
))

# 10 saniyelik hareketli ortalama
fig.add_trace(go.Scatter(
    x=df_sensor['timestamp'],
    y=df_sensor['smoothed_temp_C'],
    mode='lines',
    name='10s Rolling Average Temp (°C)',
    line=dict(color='#d9534f', width=3)
))

# 450. ve 480. saniyeler arasındaki arıza bölgesini vurgula
start_anomaly = df_sensor['timestamp'].iloc[450]
end_anomaly = df_sensor['timestamp'].iloc[480]

fig.add_vrect(
    x0=start_anomaly,
    x1=end_anomaly,
    fillcolor="red", opacity=0.2,
    layer="below", line_width=0,
    annotation_text="Thermal Spike Anomaly",
    annotation_position="top left"
)

fig.update_layout(
    title='<b>Semiconductor Stress Test: Real-Time Thermal Telemetry</b>',
    xaxis_title='Test Timestamp (Seconds)',
    yaxis_title='Core Temperature (°C)',
    template='plotly_white',
    hovermode='x unified',
    height=500
)

fig.show()
start_anomaly = '2026-11-09 10:07:30'
end_anomaly = '2026-11-09 10:08:00'
